In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
import numpy as np
import os
import torch
import torch.nn as nn
import time
import pandas as pd
from scipy.stats import pearsonr

In [21]:
from model.util import Normalizer
from model.database_util import get_hist_file, get_job_table_sample, collator
from model.model import QueryFormer
from model.database_util import Encoding
from model.dataset import PlanTreeDataset
from model.trainer import eval_workload, train

In [22]:
data_path = './data/imdb/'

In [23]:
class Args:
    # bs = 1024
    # SQ: smaller batch size
    bs = 128
    lr = 0.001
    # epochs = 200
    epochs = 100
    clip_size = 50
    embed_size = 64
    pred_hid = 128
    ffn_dim = 128
    head_size = 12
    n_layers = 8
    dropout = 0.1
    sch_decay = 0.6
    device = 'cuda:0'
    newpath = './results/full/cost/'
    to_predict = 'cost'
args = Args()

import os
if not os.path.exists(args.newpath):
    os.makedirs(args.newpath)

In [24]:
hist_file = get_hist_file(data_path + 'histogram_string.csv')
cost_norm = Normalizer(-3.61192, 12.290855)
card_norm = Normalizer(1,100)

/home/AiChaosN/Project/Phd/project/QueryFormer_VLDB2022/model/database_util.py:76: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  hist_file['freq'][i] = freq_np
/home/AiChaosN/Project/Phd/project/QueryFormer_VLDB2022/model/database_util.py:89

In [25]:
encoding_ckpt = torch.load('checkpoints/encoding.pt')
encoding = encoding_ckpt['encoding']
checkpoint = torch.load('checkpoints/cost_model.pt', map_location='cpu')

In [26]:
from model.util import seed_everything
seed_everything()

In [27]:
model = QueryFormer(emb_size = args.embed_size ,ffn_dim = args.ffn_dim, head_size = args.head_size, \
                 dropout = args.dropout, n_layers = args.n_layers, \
                 use_sample = True, use_hist = True, \
                 pred_hid = args.pred_hid
                )

In [28]:
_ = model.to(args.device)

In [29]:
to_predict = 'cost'

In [ ]:
## 训练数据集和验证数据集
imdb_path = './data/imdb/'
dfs = []  # list to hold DataFrames
# SQ: added
# for i in range(2):
for i in range(18):
    file = imdb_path + 'plan_and_cost/train_plan_part{}.csv'.format(i)
    df = pd.read_csv(file)
    dfs.append(df)

full_train_df = pd.concat(dfs)

val_dfs = []  # list to hold DataFrames
for i in range(18,20):
    file = imdb_path + 'plan_and_cost/train_plan_part{}.csv'.format(i)
    df = pd.read_csv(file)
    val_dfs.append(df)

val_df = pd.concat(val_dfs)

In [31]:
table_sample = get_job_table_sample(imdb_path+'train')

Loaded queries with len  100000
Loaded bitmaps


In [32]:
train_ds = PlanTreeDataset(full_train_df, None, encoding, hist_file, card_norm, cost_norm, to_predict, table_sample)
val_ds = PlanTreeDataset(val_df, None, encoding, hist_file, card_norm, cost_norm, to_predict, table_sample)

In [56]:
train_ds.json_df
train_ds.table_sample
train_ds.encoding
train_ds.hist_file

print(len(train_ds))
print(dir(train_ds))
x, y = train_ds[0]
print(x.keys())
print(x["x"].shape)
print(y)

print("解析df")
print(train_ds.json_df.head())

# 每个plan的cost
print("每个plan的cost")
print(len(train_ds.costs))
print(train_ds.costs[0])

# 每个plan的card
print("每个plan的card")
print(len(train_ds.cards))
print(train_ds.cards[0])

# 每个plan的encoding
print("每个plan的encoding")
print(train_ds.encoding)



10000
['__add__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'calculate_height', 'card_labels', 'cards', 'collated_dicts', 'cost_labels', 'costs', 'demo_complete_pipeline', 'demo_feature_encoding', 'demo_json_to_tree_conversion', 'demo_tree_structure_visualization', 'demo_with_sample_data', 'encoding', 'gts', 'hist_file', 'js_node2dict', 'json_df', 'labels', 'length', 'node2dict', 'old_getitem', 'pre_collate', 'table_sample', 'to_predict', 'topo_sort', 'traversePlan', 'treeNodes']
dict_keys(['x', 'attn_bias', 'rel_pos', 'heights'])
torch.Size([1, 30, 1165])
(tensor(0.6348, d

In [34]:
crit = nn.MSELoss()
model, best_path = train(model, train_ds, val_ds, crit, cost_norm, args)

Epoch: 0  Avg Loss: 3.2325574371498076e-05, Time: 5.53448224067688
Median: 1.5078207872298992
Mean: 3.4044702089261176
Training Q-error - Median: 1.5078, 90th: 5.6082, Mean: 3.4045
Epoch: 20  Avg Loss: 9.811456373427064e-06, Time: 127.99545335769653
Median: 1.225060798664774
Mean: 1.6750732784139524
Training Q-error - Median: 1.2251, 90th: 2.2858, Mean: 1.6751
Epoch: 40  Avg Loss: 7.86738691967912e-06, Time: 272.5442006587982
Median: 1.1834525585097557
Mean: 1.5597224576649449
Training Q-error - Median: 1.1835, 90th: 2.0172, Mean: 1.5597
Median: 1.1693075912739395
Mean: 1.7429181411143828
Validation Q-error - Median: 1.1693, Mean: 1.7429
Median: 1.185453837092997
Mean: 1.8157251558537697
Validation Q-error - Median: 1.1855, Mean: 1.8157
Median: 1.1822120909567222
Mean: 1.7327999950060475
Validation Q-error - Median: 1.1822, Mean: 1.7328
Median: 1.1820569036353976
Mean: 1.7597407627996755
Validation Q-error - Median: 1.1821, Mean: 1.7597
Median: 1.152220699134084
Mean: 1.717887520731009

KeyboardInterrupt: 

In [ ]:
methods = {
    'get_sample' : get_job_table_sample,
    'encoding': encoding,
    'cost_norm': cost_norm,
    'hist_file': hist_file,
    'model': model,
    'device': args.device,
    'bs': 512,
}

In [ ]:
_ = eval_workload('job-light', methods)

Loaded queries with len  70
Loaded bitmaps
Median: 1.513809230897208
Mean: 19.63840774091383
Corr:  0.8727674107264088


In [ ]:
_ = eval_workload('synthetic', methods)

Loaded queries with len  5000
Loaded bitmaps
Median: 1.1151339464064085
Mean: 1.6737023148160102
Corr:  0.982147626737304
